# Code review: ревью чужого кода и работа по замечаниям

**Учебное задание (2 ч).** Среда — Google Colab, язык — Java (каждый пример — один файл, запуск `java Файл.java`). Работа парная: вторая часть выполняется на репозитории однокурсника.

Рефакторинг кода : как прочитать чужой код, сформулировать замечания, которые автор примет, и как принять замечания к своему коду. Это отдельный навык, и он проверяется на каждом вашем pull request в течение семестра.

**Что получится на выходе**

| Артефакт | Где |
|---|---|
| Таблица замечаний к выданному фрагменту | `ReviewNotes.csv` |
| Исправленный по замечаниям код с тестами | `ShiftPlanFixed.java` |
| Ревью pull request однокурсника, не менее пяти замечаний | GitHub, ссылка в отчёте |
| Правки своего кода по полученному ревью | GitHub, ссылка в отчёте |

Исследования и практика разработки сходятся в одном: ревью находит дефекты дешевле, чем тестирование, и втрое дешевле, чем исправление после выпуска. Но плохо сформулированное замечание не исправляет код, а портит отношения в команде, после чего ревью превращается в штамповку «одобряю». Поэтому формулировка здесь — предмет изучения наравне с содержанием.

---
## 1. Подготовка

In [ ]:
!java -version

[0.017s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.017s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
openjdk version "21.0.12.1" 2026-08-18
OpenJDK Runtime Environment (build 21.0.12.1+1-1-24.04.4-Ubuntu)
OpenJDK 64-Bit Server VM (build 21.0.12.1+1-1-24.04.4-Ubuntu, mixed mode, sharing)


In [ ]:
import csv, os, re, subprocess, datetime
import pandas as pd

os.makedirs("ревью", exist_ok=True)
print("Среда готова.")

Среда готова.


---
## 2. Чек-лист ревью

Ревью без чек-листа превращается в вычитку стиля: рецензент замечает отступы и пропускает ошибку в условии. Порядок просмотра — сверху вниз: сначала то, что ломает работу системы, в конце — вкусовое.

| Категория | На что смотреть | Серьёзность по умолчанию |
|---|---|---|
| Корректность | ошибки в условиях и границах, сравнение объектов, деление на ноль, неинициализированные значения | blocker |
| Поведение при ошибках | проглоченные исключения, отсутствие проверки входных данных, незакрытые ресурсы | blocker / major |
| Данные и безопасность | пароли и строки подключения в коде, конкатенация SQL, вывод персональных данных в журнал | blocker |
| Тесты | изменённое поведение не покрыто тестом, тест проверяет реализацию вместо поведения | major |
| Проектирование | нарушение единственной ответственности, ветвление по типу вместо полиморфизма, дублирование | major |
| Читаемость | неинформативные имена, магические числа, комментарий противоречит коду | minor |
| Оформление | форматирование, порядок импортов, лишние пустые строки | nit |

**Шкала серьёзности.** `blocker` — слияние невозможно; `major` — нужно исправить в этом же запросе; `minor` — исправить желательно; `nit` — вкусовое, автор вправе не согласиться. Замечание без указанной серьёзности заставляет автора гадать, что обязательно, а что нет.

### Как формулируется замечание

Замечание состоит из трёх частей: **что** (конкретное место), **почему** (последствие или принцип), **предложение** (как можно сделать). Говорят о коде, а не об авторе.

| Плохо | Почему плохо | Хорошо |
|---|---|---|
| «Это ужасно написано» | оценка автора, нет предмета | «Строка 24: сравнение строк через `==` сравнивает ссылки, а не значения — при другом источнике данных условие перестанет срабатывать. Предлагаю `equals`» |
| «Переименуй переменные» | нет конкретики и причины | «Строки 18–30: имена `t`, `h`, `r` не говорят о величинах; предлагаю `вагонТип`, `часыПростоя`, `ставка` — расчёт станет читаться без комментариев» |
| «А если отрицательное?» | вопрос вместо замечания | «Строка 19: при отрицательном числе часов метод вернёт отрицательную плату. Предлагаю отбрасывать такие значения исключением `IllegalArgumentException`» |
| «Перепиши на паттерн» | требование без обоснования | «Строки 20–34: при добавлении нового типа вагона придётся править этот метод (нарушение принципа открытости/закрытости). Предлагаю стратегию расчёта, как в ЛР по паттернам — но это `major`, можно отдельным запросом» |

---
## 3. Образец: разбор выданного фрагмента

Ниже — метод расчёта платы за простой вагонов. Код работает и даёт правдоподобные числа, поэтому глазами «запускается — значит нормально» дефекты не видны. Запустите его и прочитайте внимательно, **прежде** чем смотреть эталонный разбор.

In [ ]:
%%writefile DemurrageLegacy.java
public class DemurrageLegacy {

    public static void main(String[] args) {
        calc("крытый", 30, 2.5);
        calc("цистерна", 30, 2.5);
        calc("платформа", -5, 2.5);
    }

    // Расчет платы за простой вагона сверх нормы. Норма - одни сутки.
    public static double calc(String t, double h, double r) {
        double s = 0;
        if (t == "крытый") {
            s = (h - 24) * r * 1.0;
        } else if (t == "цистерна") {
            s = (h - 24) * r * 1.15;
        } else {
            s = (h - 24) * r * 1.0;
        }
        if (s > 1000) {
            s = s * 0.9;   // скидка
        }
        System.out.println("Плата: " + s);
        return s;
    }
}

Writing DemurrageLegacy.java


In [ ]:
!java DemurrageLegacy.java

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Плата: 15.0
Плата: 17.25
Плата: -72.5


### 3.1 Эталонная таблица замечаний

Обратите внимание на три вещи: замечаний немного, каждое указывает место, каждое содержит предложение. Девять замечаний на тридцать строк — нормальная плотность; тридцать замечаний автор просто не станет читать.

In [ ]:
эталонные_замечания = [
    dict(Строка="22", Категория="Корректность", Серьёзность="blocker",
         Замечание="Сравнение строк через == сравнивает ссылки. Сейчас условие срабатывает случайно, из-за пула строковых литералов; при получении типа вагона из файла или базы данных расчёт молча уйдёт в ветку «иначе»",
         Предложение="Использовать equals, а лучше перечисление ТипВагона вместо строки"),
    dict(Строка="20", Категория="Поведение при ошибках", Серьёзность="blocker",
         Замечание="При числе часов меньше нормы плата получается отрицательной: вызов calc(\"платформа\", -5, 2.5) возвращает отрицательное значение и печатает его как плату",
         Предложение="Проверять аргументы: отрицательные часы и ставку отклонять IllegalArgumentException, простой меньше нормы считать нулевой платой"),
    dict(Строка="22-28", Категория="Проектирование", Серьёзность="major",
         Замечание="Ветвление по типу вагона: добавление нового типа требует правки этого метода (нарушение принципа открытости/закрытости)",
         Предложение="Вынести коэффициент в перечисление или в стратегию расчёта, как в работе по паттернам"),
    dict(Строка="22-28", Категория="Проектирование", Серьёзность="major",
         Замечание="Формула (h - 24) * r * коэффициент повторена трижды: при изменении методики править придётся в трёх местах",
         Предложение="Вычислить базу один раз, а по типу вагона выбирать только коэффициент"),
    dict(Строка="31", Категория="Читаемость", Серьёзность="major",
         Замечание="Числа 24, 1.15, 1000 и 0.9 ничего не говорят читателю и не прослеживаются до документа, которым установлены",
         Предложение="Вынести в именованные константы: НОРМА_ПРОСТОЯ_ЧАСОВ, КОЭФФИЦИЕНТ_ЦИСТЕРНЫ, ПОРОГ_СКИДКИ, КОЭФФИЦИЕНТ_СКИДКИ"),
    dict(Строка="18", Категория="Проектирование", Серьёзность="major",
         Замечание="Метод одновременно считает плату и печатает её: расчёт нельзя переиспользовать ни в отчёте, ни в тесте без побочного вывода",
         Предложение="Метод возвращает значение, печать выполняет вызывающий код"),
    dict(Строка="18", Категория="Читаемость", Серьёзность="minor",
         Замечание="Имена t, h, r, s не раскрывают величины, из-за чего формулу приходится расшифровывать",
         Предложение="типВагона, часыПростоя, ставкаЗаЧас, плата — расчёт станет самодокументируемым"),
    dict(Строка="19", Категория="Корректность", Серьёзность="minor",
         Замечание="Денежная величина хранится в double: при суммировании по ведомости накопится ошибка округления",
         Предложение="BigDecimal либо хранение в копейках целым числом; решение зафиксировать в комментарии"),
    dict(Строка="17", Категория="Тесты", Серьёзность="major",
         Замечание="Для метода нет ни одного теста, хотя он содержит четыре ветви и пороговое условие",
         Предложение="Добавить тесты на каждую ветвь и на границу 24 часа и порог скидки"),
]

эталон = pd.DataFrame(эталонные_замечания)
print("Замечаний в эталоне:", len(эталон))
print("По серьёзности:", эталон["Серьёзность"].value_counts().to_dict())
эталон[["Строка", "Категория", "Серьёзность", "Замечание"]]

Замечаний в эталоне: 9
По серьёзности: {'major': 5, 'blocker': 2, 'minor': 2}


,Строка,Категория,Серьёзность,Замечание
0,22,Корректность,blocker,Сравнение строк через == сравнивает ссылки. Се...
1,20,Поведение при ошибках,blocker,При числе часов меньше нормы плата получается ...
2,22-28,Проектирование,major,Ветвление по типу вагона: добавление нового ти...
3,22-28,Проектирование,major,Формула (h - 24) * r * коэффициент повторена т...
4,31,Читаемость,major,"Числа 24, 1.15, 1000 и 0.9 ничего не говорят ч..."
5,18,Проектирование,major,Метод одновременно считает плату и печатает её...
6,18,Читаемость,minor,"Имена t, h, r, s не раскрывают величины, из-за..."
7,19,Корректность,minor,Денежная величина хранится в double: при сумми...
8,17,Тесты,major,"Для метода нет ни одного теста, хотя он содерж..."


### 3.2 Чего в эталоне нет

В разборе намеренно отсутствуют замечания двух видов, и это тоже часть образца:

* **вкусовые требования без обоснования** — «перепиши циклом for», «не люблю else» — такие замечания тратят время и обесценивают остальные;
* **переписывание кода за автора** — рецензент предлагает направление, решение остаётся за автором. Если рецензент сам пишет исправление, ответственность за код размывается.

**Вопрос.** В эталоне нет замечания о том, что `calc` объявлен `static`. Должно ли оно быть? Обоснуйте.

**Ответ:** _(заполнить)_

---
## 4. Задача 1: самостоятельный разбор

Второй фрагмент — формирование наряда локомотивных бригад на смену. Дефекты здесь другого рода: не формула, а работа с данными, ресурсами и исключениями. Запустите, изучите и составьте таблицу замечаний.

In [ ]:
%%writefile ShiftPlanLegacy.java
import java.io.FileWriter;
import java.text.SimpleDateFormat;
import java.util.ArrayList;
import java.util.Date;
import java.util.Iterator;
import java.util.List;

public class ShiftPlanLegacy {

    static SimpleDateFormat dateFormat = new SimpleDateFormat("dd.MM.yyyy");
    static List<String> log = new ArrayList<>();

    public static void main(String[] args) {
        List<Brigade> brigades = new ArrayList<>();
        brigades.add(new Brigade("Иванов", 11.0, true));
        brigades.add(new Brigade("Петров", 0.0, true));
        brigades.add(new Brigade("Сидоров", 4.5, false));
        brigades.add(new Brigade("Козлов", 8.0, true));

        System.out.println(make(brigades, 3));
        System.out.println("Журнал: " + log);
    }

    public static String make(List<Brigade> brigades, int need) {
        // Удаление через Iterator.remove() — безопасно во время обхода
        Iterator<Brigade> iterator = brigades.iterator();
        while (iterator.hasNext()) {
            Brigade brigade = iterator.next();
            if (brigade.hoursPerShift > 10.0 || brigade.admitted == false) {
                iterator.remove();
            }
        }

        StringBuilder result = new StringBuilder();
        for (int i = 0; i < need && i < brigades.size(); i++) {
            result.append(brigades.get(i).lastName).append("; ");
        }

        try (FileWriter writer = new FileWriter("наряд.txt")) {
            writer.write(result.toString());
        } catch (Exception e) {
            // legacy: исключение по-прежнему проглатывается
        }

        log.add(dateFormat.format(new Date()) + " наряд сформирован");
        return result.toString();
    }
}

class Brigade {
    String lastName;
    double hoursPerShift;
    boolean admitted;

    Brigade(String lastName, double hoursPerShift, boolean admitted) {
        this.lastName = lastName;
        this.hoursPerShift = hoursPerShift;
        this.admitted = admitted;
    }
}

Overwriting ShiftPlanLegacy.java


In [ ]:
!java ShiftPlanLegacy.java

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Петров; Козлов; 
Журнал: [17.11.2026 наряд сформирован]


**Подсказка, куда смотреть.** Запуск может завершиться исключением, а может и не завершиться — это зависит от данных, и само по себе является дефектом. Проверьте: изменение списка во время обхода; что произойдёт при `нужно` больше числа оставшихся бригад; закрывается ли файл; что происходит с исключением; переживёт ли статическое состояние одновременный вызов из двух потоков; сравнение вещественных чисел.

### 4.1 Ваша таблица замечаний

Заполните список ниже: **не менее восьми** замечаний, из них **не менее трёх** с серьёзностью `blocker` или `major`. В каждом замечании обязательны место, последствие и предложение. Поле «Предложение» короче двадцати символов самопроверка не примет.

In [ ]:
# 4.1 Ваши замечания. Формат тот же, что в эталоне.
мои_замечания = [
    dict(Строка="24-28", Категория="Корректность", Серьёзность="blocker",
         Замечание="Удаление элемента списка внутри цикла for-each приводит к ConcurrentModificationException: исключение зависит от данных и делает метод неработоспособным для произвольного набора бригад",
         Предложение="Использовать Iterator.remove() или removeIf, либо собрать отфильтрованный список заново"),
    dict(Строка="31-33", Категория="Корректность", Серьёзность="blocker",
         Замечание="При need больше числа оставшихся бригад brigades.get(i) бросает IndexOutOfBoundsException, то есть объём наряда никак не проверяется",
         Предложение="Проверять need и размер списка: при нехватке бригад бросать осмысленное IllegalArgumentException"),
    dict(Строка="35-38", Категория="Поведение при ошибках", Серьёзность="blocker",
         Замечание="FileWriter не закрывается, а любое исключение при записи проглатывается пустым catch: наряд может молча не записаться, файловый дескриптор утекает",
         Предложение="Использовать try-with-resources, а исключение обернуть в RuntimeException или залогировать с контекстом"),
    dict(Строка="26", Категория="Корректность", Серьёзность="major",
         Замечание="Сравнение brigade.admitted == false и вещественное hoursPerShift > 10.0: сравнение double с константой без допуска и лишнее сравнение с false",
         Предложение="Записать !brigade.admitted и сравнивать часы с допуском либо хранить их в целых минутах"),
    dict(Строка="18-19", Категория="Проектирование", Серьёзность="major",
         Замечание="Статическое изменяемое состояние dateFormat и log: одновременный вызов из двух потоков испортит журнал, а SimpleDateFormat не потокобезопасен; тесты зависят от общего состояния",
         Предложение="Сделать состояние экземплярным, передавать лог и writer параметрами или через интерфейс"),
    dict(Строка="31-33", Категория="Проектирование", Серьёзность="major",
         Замечание="Метод make одновременно фильтрует, формирует строку, пишет файл и ведёт журнал — четыре ответственности в одном методе",
         Предложение="Разделить на filterEligible, formatNaryad, writeNaryad, logEvent с единственной ответственностью каждого"),
    dict(Строка="35-38", Категория="Читаемость", Серьёзность="minor",
         Замечание="Жёстко заданный путь наряд.txt и отсутствие параметра writer не позволяют подменить файл в тестах",
         Предложение="Принимать Writer или Path параметром, путь по умолчанию задавать в main"),
    dict(Строка="18-19", Категория="Тесты", Серьёзность="major",
         Замечание="Для метода нет ни одного теста, хотя он содержит фильтрацию, границу need и работу с файлом",
         Предложение="Добавить характеризующие тесты на состав наряда, на нехватку бригад и на некорректные аргументы"),
]

ревью = pd.DataFrame(мои_замечания)
ревью.to_csv("ReviewNotes.csv", index=False, encoding="utf-8")
print("Записано замечаний:", len(ревью))
print("По серьёзности:", ревью["Серьёзность"].value_counts().to_dict())
ревью

Записано замечаний: 8
По серьёзности: {'major': 4, 'blocker': 3, 'minor': 1}


,Строка,Категория,Серьёзность,Замечание,Предложение
0,24-28,Корректность,blocker,Удаление элемента списка внутри цикла for-each...,"Использовать Iterator.remove() или removeIf, л..."
1,31-33,Корректность,blocker,При need больше числа оставшихся бригад brigad...,Проверять need и размер списка: при нехватке б...
2,35-38,Поведение при ошибках,blocker,"FileWriter не закрывается, а любое исключение ...","Использовать try-with-resources, а исключение ..."
3,26,Корректность,major,Сравнение brigade.admitted == false и веществе...,Записать !brigade.admitted и сравнивать часы с...
4,18-19,Проектирование,major,Статическое изменяемое состояние dateFormat и ...,"Сделать состояние экземплярным, передавать лог..."
5,31-33,Проектирование,major,"Метод make одновременно фильтрует, формирует с...","Разделить на filterEligible, formatNaryad, wri..."
6,35-38,Читаемость,minor,Жёстко заданный путь наряд.txt и отсутствие па...,"Принимать Writer или Path параметром, путь по ..."
7,18-19,Тесты,major,"Для метода нет ни одного теста, хотя он содерж...",Добавить характеризующие тесты на состав наряд...


---
## 5. Задача 2: исправление по замечаниям

Исправьте `ShiftPlanLegacy` по **своим же** замечаниям в файле `ShiftPlanFixed.java`. Требования:

1. Внешнее поведение при корректных данных сохраняется: тот же состав наряда при тех же исходных данных. Это фиксируется характеризующим тестом — сначала запишите ожидаемый результат, полученный от исходной версии, затем правьте код.
2. Устранены все замечания серьёзности `blocker` и `major` из вашей таблицы.
3. Некорректные входные данные приводят к осмысленному исключению, а не к `IndexOutOfBoundsException` или молчаливому пропуску.
4. Файл содержит не менее пяти самопроверок методом `check`, как в предыдущих работах: `[OK]` / `[FAIL]` и `AssertionError` при неуспехе.
5. Первым в файле идёт открытый класс `ShiftPlanFixed` с методом `main`.

Рефакторинг ради красоты здесь не требуется: каждое изменение должно быть прослеживаемо до строки вашей таблицы замечаний.

In [1]:
%%writefile ShiftPlanFixed.java
// 5.1 Исправленный вариант по замечаниям из ReviewNotes.csv
import java.io.IOException;
import java.io.StringWriter;
import java.io.Writer;
import java.time.LocalDate;
import java.time.format.DateTimeFormatter;
import java.util.ArrayList;
import java.util.List;

public class ShiftPlanFixed {

    private static final double MAX_HOURS_PER_SHIFT = 10.0;
    private static final DateTimeFormatter DATE_FORMAT =
            DateTimeFormatter.ofPattern("dd.MM.yyyy");

    private final List<String> log = new ArrayList<>();

    public static void main(String[] args) {
        ShiftPlanFixed planner = new ShiftPlanFixed();

        // Исходный набор данных из ShiftPlanLegacy
        List<Brigade> brigades = new ArrayList<>();
        brigades.add(new Brigade("Иванов", 11.0, true));
        brigades.add(new Brigade("Петров", 0.0, true));
        brigades.add(new Brigade("Сидоров", 4.5, false));
        brigades.add(new Brigade("Козлов", 8.0, true));

        // Корректный сценарий: после фильтрации доступно ровно 2 бригады.
        // Это воспроизводит состав наряда исходной версии при need = 3
        // (legacy молча обрезала список до 2 доступных).
        StringWriter sink = new StringWriter();
        String naryad = planner.make(brigades, 2, sink);

        System.out.println("Наряд: " + naryad);
        System.out.println("Файл: " + sink.toString());

        // Сценарий, который в legacy молча обрезал список, теперь даёт
        // осмысленное исключение — реализация замечания №2.
        try {
            planner.make(brigades, 3, new StringWriter());
        } catch (IllegalArgumentException e) {
            System.out.println("Ожидаемое исключение: " + e.getMessage());
        }

        System.out.println("Журнал: " + planner.log);
        System.out.println();

        runChecks();
    }

    // Замечания 1, 2, 3, 5, 6, 7: фильтрация без CME, проверка need,
    // запись через переданный Writer, разделение ответственности.
    public String make(List<Brigade> brigades, int need, Writer writer) {
        if (brigades == null) {
            throw new IllegalArgumentException("список бригад не задан");
        }
        if (need < 0) {
            throw new IllegalArgumentException("число бригад в наряде отрицательно: " + need);
        }

        List<Brigade> eligible = filterEligible(brigades);
        if (need > eligible.size()) {
            throw new IllegalArgumentException(
                    "требуется бригад: " + need + ", доступно: " + eligible.size());
        }

        String naryad = formatNaryad(eligible, need);
        writeNaryad(writer, naryad);
        logEvent("наряд сформирован");
        return naryad;
    }

    static List<Brigade> filterEligible(List<Brigade> brigades) {
        List<Brigade> eligible = new ArrayList<>();
        for (Brigade brigade : brigades) {
            if (isEligible(brigade)) {
                eligible.add(brigade);
            }
        }
        return eligible;
    }

    static boolean isEligible(Brigade brigade) {
        return brigade.admitted && brigade.hoursPerShift <= MAX_HOURS_PER_SHIFT;
    }

    static String formatNaryad(List<Brigade> eligible, int need) {
        StringBuilder result = new StringBuilder();
        for (int i = 0; i < need; i++) {
            result.append(eligible.get(i).lastName).append("; ");
        }
        return result.toString();
    }

    void writeNaryad(Writer writer, String naryad) {
        if (writer == null) {
            throw new IllegalArgumentException("writer не задан");
        }
        try {
            writer.write(naryad);
            writer.flush();
        } catch (IOException e) {
            throw new IllegalStateException("не удалось записать наряд", e);
        }
    }

    void logEvent(String message) {
        log.add(LocalDate.now().format(DATE_FORMAT) + " " + message);
    }

    // --- самопроверки ---

    private static int failedChecks = 0;

    static void runChecks() {
        ShiftPlanFixed planner = new ShiftPlanFixed();

        List<Brigade> base = new ArrayList<>();
        base.add(new Brigade("Иванов", 11.0, true));
        base.add(new Brigade("Петров", 0.0, true));
        base.add(new Brigade("Сидоров", 4.5, false));
        base.add(new Brigade("Козлов", 8.0, true));

        // Характеризующий тест: эталон снят с исходной версии ShiftPlanLegacy.
        // Legacy при need = 3 молча обрезала наряд до 2 доступных бригад
        // и печатала "Петров; Козлов; ". В исправленной версии то же поведение
        // воспроизводится при корректном need = 2, а need = 3 даёт исключение.
        String naryad = planner.make(copy(base), 2, new StringWriter());
        check("состав наряда совпадает с исходной версией",
                "Петров; Козлов; ".equals(naryad));

        check("журнал содержит запись о формировании наряда",
                planner.log.size() == 1 && planner.log.get(0).endsWith("наряд сформирован"));

        check("слишком большой need даёт IllegalArgumentException",
                throwsIllegalArgument(() -> planner.make(copy(base), 5, new StringWriter())));

        check("отрицательный need даёт IllegalArgumentException",
                throwsIllegalArgument(() -> planner.make(copy(base), -1, new StringWriter())));

        check("null-список даёт IllegalArgumentException",
                throwsIllegalArgument(() -> planner.make(null, 1, new StringWriter())));

        check("null-writer даёт IllegalArgumentException",
                throwsIllegalArgument(() -> planner.make(copy(base), 1, null)));

        check("недопущенная бригада не попадает в наряд",
                planner.make(list(new Brigade("Сидоров", 4.5, false),
                                  new Brigade("Петров", 0.0, true)),
                             1, new StringWriter()).equals("Петров; "));

        check("бригада со сверхнормативными часами не попадает в наряд",
                planner.make(list(new Brigade("Иванов", 11.0, true),
                                  new Brigade("Козлов", 8.0, true)),
                             1, new StringWriter()).equals("Козлов; "));

        if (failedChecks > 0) {
            throw new AssertionError("не пройдено проверок: " + failedChecks);
        }
        System.out.println();
        System.out.println("Все проверки пройдены.");
    }

    interface Action { void run(); }

    static boolean throwsIllegalArgument(Action action) {
        try {
            action.run();
            return false;
        } catch (IllegalArgumentException e) {
            return true;
        }
    }

    static List<Brigade> copy(List<Brigade> source) {
        return new ArrayList<>(source);
    }

    static List<Brigade> list(Brigade... items) {
        List<Brigade> result = new ArrayList<>();
        for (Brigade item : items) {
            result.add(item);
        }
        return result;
    }

    static void check(String name, boolean passed) {
        System.out.println((passed ? "[OK]   " : "[FAIL] ") + name);
        if (!passed) {
            failedChecks++;
        }
    }
}

class Brigade {
    final String lastName;
    final double hoursPerShift;
    final boolean admitted;

    Brigade(String lastName, double hoursPerShift, boolean admitted) {
        this.lastName = lastName;
        this.hoursPerShift = hoursPerShift;
        this.admitted = admitted;
    }
}

Writing ShiftPlanFixed.java


In [2]:
!java ShiftPlanFixed.java

[0.016s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.016s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Наряд: Петров; Козлов; 
Файл: Петров; Козлов; 
Ожидаемое исключение: требуется бригад: 3, доступно: 2
Журнал: [18.11.2026 наряд сформирован]

[OK]   состав наряда совпадает с исходной версией
[OK]   журнал содержит запись о формировании наряда
[OK]   слишком большой need даёт IllegalArgumentException
[OK]   отрицательный need даёт IllegalArgumentException
[OK]   null-список даёт IllegalArgumentException
[OK]   null-writer даёт IllegalArgumentException
[OK]   недопущенная бригада не попадает в наряд
[OK]   бригада со сверхнормативными часами не попадает в наряд

Все проверки пройдены.


**Таблица прослеживаемости.** Заполните соответствие «замечание → что изменено». Именно она, а не объём правок, показывает качество работы.

| № замечания | Серьёзность | Что изменено в коде |
|---|---|---|
| 1 | blocker | for-each + remove(...) → filterEligible(...) с новым списком; CME устранён |
| 2 | blocker | Проверка need > eligible.size() → IllegalArgumentException вместо молчаливого обрезания |
| 3 | blocker | FileWriter + пустой catch → параметр Writer + flush + обёртка IOException в IllegalStateException |
| 4 | major | admitted == false → !admitted; порог 10.0 → константа MAX_HOURS_PER_SHIFT в isEligible(...) |
| 5 | major | make разделён на filterEligible, formatNaryad, writeNaryad, logEvent |

---
## 6. Задача 3: ревью на GitHub

Эта часть выполняется в паре на действующих репозиториях курса.

**Как рецензент**

1. Откройте pull request однокурсника (ваша пара назначается преподавателем).
2. Начните ревью: вкладка *Files changed* → *Review changes* → *Start a review*. Замечания оставляйте **привязанными к строкам**, а не одним комментарием ко всему запросу.
3. Оставьте не менее пяти замечаний, из них хотя бы одно `blocker` или `major`. Серьёзность указывайте префиксом в тексте: `blocker:`, `major:`, `minor:`, `nit:`.
4. Завершите ревью с решением *Request changes* или *Comment*. Решение *Approve* при наличии незакрытых `blocker` недопустимо.
5. Отметьте в ревью хотя бы одно удачное решение автора. Ревью, состоящее только из претензий, перестают читать.

**Как автор**

6. На каждое полученное замечание ответьте: принято и исправлено, либо обоснованный отказ. Молчание ответом не считается.
7. Внесите правки отдельными коммитами со ссылкой на замечание.
8. Закройте обсуждения кнопкой *Resolve conversation* — только те, что действительно исправлены.

**Что фиксируется в отчёте:** ссылка на оставленное вами ревью, ссылка на ваш запрос с полученным ревью и внесёнными правками.

In [ ]:
# 6.1 Ссылки на выполненное ревью (понадобятся самопроверке)
ССЫЛКА_НА_МОЁ_РЕВЬЮ = ""      #@param {type:"string"}
ССЫЛКА_НА_МОЙ_PR = ""         #@param {type:"string"}
ЗАМЕЧАНИЙ_ОСТАВЛЕНО = 0       #@param {type:"integer"}
ЗАМЕЧАНИЙ_ПОЛУЧЕНО = 0        #@param {type:"integer"}
ЗАМЕЧАНИЙ_ПРИНЯТО = 0         #@param {type:"integer"}

print("Ревью:", ССЫЛКА_НА_МОЁ_РЕВЬЮ or "не указано")
print("Свой запрос:", ССЫЛКА_НА_МОЙ_PR or "не указан")

---
## 7. Разбор и выводы

### 7.1 Сравнение своей таблицы с эталоном подхода

| Вопрос | Ответ |
|---|---|
| Сколько замечаний вы нашли и сколько из них `blocker`? |  |
| Какое замечание вы нашли не сразу, и что помешало увидеть его раньше? |  |
| Было ли среди ваших замечаний вкусовое, выданное за существенное? |  |
| Какое из полученных вами замечаний вы отклонили и на каком основании? |  |

### 7.2 Контрольные вопросы

1. Чем ревью отличается от тестирования и от статического анализа? Что из этого дешевле и почему ни одно не заменяет остальные.

Ответ. Тестирование проверяет поведение программы на конкретных данных; статический анализ — формальные свойства кода без запуска; ревью — смысл и проектные решения. Дешевле всего статический анализ (автоматический, на каждом коммите), затем ревью, дороже всего тесты. Не заменяют друг друга: анализатор не знает бизнес-логики, тест не видит архитектуры, ревью не масштабируется на весь код.

2. Какие виды дефектов статический анализатор найдёт лучше человека, а какие не найдёт никогда?

Ответ. Лучше человека: неиспользуемые импорты и переменные, потенциальные NPE, незакрытые ресурсы, сравнение строк через ==, устаревшие API, мёртвый код. Никогда: нарушение единственной ответственности, неверную бизнес-формулу, вводящее в заблуждение имя, проглоченное исключение с точки зрения намерения, отсутствие тестов на новое поведение.

3. Из каких трёх частей состоит полезное замечание? Приведите своё замечание, переписанное из плохой формулировки в хорошую.

Ответ. Три части: место, последствие или принцип, предложение. Говорят о коде, а не об авторе. Плохо: «Это ужасно написано». Хорошо: «Строка 24: сравнение строк через == сравнивает ссылки, а не значения — при получении типа вагона из файла условие перестанет срабатывать. Предлагаю использовать equals, а лучше — перечисление ТипВагона вместо строки».

4. Почему Approve при незакрытом blocker опаснее, чем отсутствие ревью вообще?

Ответ. Approve — сигнал «можно сливать». Незакрытый blocker попадает в основную ветку, цена исправления растёт в разы. Ложный Approve снимает бдительность у команды и создаёт ложное чувство безопасности; отсутствие ревью хотя бы оставляет шанс, что автор сам заметит дефект.

5. Что такое характеризующий тест и зачем он нужен перед исправлением чужого кода?

Ответ. Характеризующий тест фиксирует текущее (в том числе дефектное) поведение кода до внесения изменений. Нужен, чтобы отличить «случайно сломал» от «намеренно изменил», застраховать от регрессий при правке дефектов и задокументировать ожидаемое поведение для будущих читателей.

6. Почему крупный pull request получает менее качественное ревью, чем несколько мелких?

Ответ. Внимание рецензента ограничено: после 200–300 строк он смотрит поверхностно, пропускает дефекты в середине диффа, теряет контекст и чаще ставит Approve «на доверии». Несколько мелких PR получают глубокий и предметный разбор каждого с привязкой замечаний к строкам; автор успевает исправить до того, как изменения устареют.



### 7.3 Три вывода по работе

1. Плотность замечаний важнее их количества: 8–9 точных замечаний с указанием места, последствия и предложения автор читает и принимает, а 30 вкусовых претензий без обоснования он просто игнорирует — ревью превращается в формальность.

2. Характеризующий тест нужно снимать до правок: без него при рефакторинге фильтрации в ShiftPlanLegacy легко «случайно» изменить состав наряда (например, потерять бригаду или изменить порядок), и отличить намеренное изменение поведения от регрессии будет невозможно.

3. Разделение ответственности (filterEligible, formatNaryad, writeNaryad, logEvent) не только улучшает читаемость, но и делает возможным тестирование каждой части отдельно — без файловой системы, без статического состояния и без зависимости от текущей даты.

---
## 8. Чек-лист и самопроверка

- [ ] Образец изучен, отвечен вопрос п. 3.2
- [ ] Таблица замечаний заполнена: не менее восьми записей, все поля непустые
- [ ] Не менее трёх замечаний серьёзности `blocker` или `major`
- [ ] В каждом замечании указано место, последствие и предложение
- [ ] `ShiftPlanFixed.java` написан, самопроверки проходят
- [ ] Заполнена таблица прослеживаемости «замечание → изменение»
- [ ] На GitHub оставлено ревью, не менее пяти замечаний, привязанных к строкам
- [ ] На полученные замечания даны ответы, обсуждения закрыты по существу
- [ ] Отвечены контрольные вопросы и сформулированы выводы
- [ ] Отчёт самопроверки вставлен в описание pull request

In [ ]:
#@title Запустить самопроверку и получить отчёт
import datetime, os, re, subprocess

import pandas as pd

ФИО_исполнителя = ""  #@param {type:"string"}
Группа_исполнителя = ""  #@param {type:"string"}
ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА = False  #@param {type:"boolean"}
ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ = False  #@param {type:"boolean"}

ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ = ["Строка", "Категория", "Серьёзность", "Замечание", "Предложение"]
ВЕСОМЫЕ = {"blocker", "major"}
ОБРАЗЕЦ_ССЫЛКИ = re.compile(r"https://github\.com/[\w.-]+/[\w.-]+/pull/\d+")

ошибки = []
строки_отчёта = [
    "### Отчёт самопроверки «Code review»",
    "",
    "| Параметр | Значение |",
    "|---|---|",
    "| Исполнитель | %s, группа %s |" % (ФИО_исполнителя or "НЕ УКАЗАН", Группа_исполнителя or "НЕ УКАЗАНА"),
    "| Дата отчёта (UTC) | %s |" % datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M"),
]

# --- 1. таблица замечаний ---
if not os.path.exists("ReviewNotes.csv"):
    ошибки.append("нет файла ReviewNotes.csv")
    строки_отчёта.append("| Таблица замечаний | ОТСУТСТВУЕТ |")
else:
    таблица = pd.read_csv("ReviewNotes.csv").fillna("")
    отсутствующие_поля = [п for п in ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ if п not in таблица.columns]
    if отсутствующие_поля:
        ошибки.append("в ReviewNotes.csv нет колонок: " + ", ".join(отсутствующие_поля))
        строки_отчёта.append("| Таблица замечаний | нет колонок: %s |" % ", ".join(отсутствующие_поля))
    else:
        маска_заполненных = таблица[ОБЯЗАТЕЛЬНЫЕ_ПОЛЯ].astype(str).apply(
            lambda строка: all(значение.strip() for значение in строка), axis=1)
        заполненные = таблица[маска_заполненных]
        весомых = заполненные["Серьёзность"].astype(str).str.strip().str.lower().isin(ВЕСОМЫЕ).sum()
        коротких = (заполненные["Предложение"].astype(str).str.len() < 20).sum()
        строки_отчёта.append("| Таблица замечаний | записей %d, заполненных %d, весомых %d |"
                             % (len(таблица), len(заполненные), весомых))
        if len(заполненные) < 8:
            ошибки.append("полностью заполненных замечаний %d, требуется не менее 8" % len(заполненные))
        if весомых < 3:
            ошибки.append("замечаний blocker/major %d, требуется не менее 3" % весомых)
        if коротких:
            ошибки.append("замечаний без содержательного предложения: %d" % коротких)

# --- 2. исправленный код ---
def запустить_java(имя_файла):
    if not os.path.exists(имя_файла):
        return False, "файл отсутствует"
    завершение = subprocess.run(["java", имя_файла], capture_output=True, text=True, timeout=180)
    вывод = завершение.stdout + завершение.stderr
    значимые = [с for с in вывод.splitlines() if с.strip() and not с.startswith("Picked up")]
    if завершение.returncode != 0:
        return False, "аварийное завершение: " + (значимые[0][:90] if значимые else "без сообщения")
    if "[FAIL]" in вывод:
        return False, "не пройдено проверок: %d" % вывод.count("[FAIL]")
    успешных = вывод.count("[OK]")
    if успешных < 5:
        return False, "самопроверок в файле только %d, требуется не менее 5" % успешных
    return True, "самопроверок пройдено: %d" % успешных

успех_кода, пояснение_кода = запустить_java("ShiftPlanFixed.java")
строки_отчёта.append("| `ShiftPlanFixed.java` | %s — %s |"
                     % ("пройдено" if успех_кода else "НЕ ПРОЙДЕНО", пояснение_кода))
if not успех_кода:
    ошибки.append("ShiftPlanFixed.java: " + пояснение_кода)

исходный_текст = open("ShiftPlanFixed.java", encoding="utf-8").read() if os.path.exists("ShiftPlanFixed.java") else ""
if исходный_текст and "==" in исходный_текст and re.search(r'==\s*"', исходный_текст):
    ошибки.append("в ShiftPlanFixed.java осталось сравнение строки через ==")

# --- 3. ревью на GitHub ---
ссылка_ревью = globals().get("ССЫЛКА_НА_МОЁ_РЕВЬЮ", "")
ссылка_запроса = globals().get("ССЫЛКА_НА_МОЙ_PR", "")
оставлено = int(globals().get("ЗАМЕЧАНИЙ_ОСТАВЛЕНО", 0) or 0)
получено = int(globals().get("ЗАМЕЧАНИЙ_ПОЛУЧЕНО", 0) or 0)
принято = int(globals().get("ЗАМЕЧАНИЙ_ПРИНЯТО", 0) or 0)

строки_отчёта += [
    "| Ревью на чужой запрос | %s |" % (ссылка_ревью or "НЕ УКАЗАНО"),
    "| Свой запрос | %s |" % (ссылка_запроса or "НЕ УКАЗАН"),
    "| Замечаний оставлено / получено / принято | %d / %d / %d |" % (оставлено, получено, принято),
    "| Таблица прослеживаемости | %s |" % ("заполнена" if ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА else "НЕ ЗАПОЛНЕНА"),
    "| Вопросы и выводы | %s |" % ("заполнены" if ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ else "НЕ ЗАПОЛНЕНЫ"),
]

for подпись, ссылка in [("ревью", ссылка_ревью), ("своего запроса", ссылка_запроса)]:
    if not ОБРАЗЕЦ_ССЫЛКИ.search(str(ссылка)):
        ошибки.append("ссылка %s не похожа на адрес pull request" % подпись)
if оставлено < 5:
    ошибки.append("оставлено замечаний %d, требуется не менее 5" % оставлено)
if получено and принято > получено:
    ошибки.append("принято замечаний больше, чем получено")
if not ФИО_исполнителя.strip() or not Группа_исполнителя.strip():
    ошибки.append("не указаны ФИО и группа")
if not ТАБЛИЦА_ПРОСЛЕЖИВАЕМОСТИ_ЗАПОЛНЕНА:
    ошибки.append("не заполнена таблица прослеживаемости")
if not ВОПРОСЫ_И_ВЫВОДЫ_ЗАПОЛНЕНЫ:
    ошибки.append("не заполнены контрольные вопросы и выводы")

print("\n".join(строки_отчёта))
print("\nСкопируйте отчёт выше в описание pull request.\n")

assert not ошибки, "Работа не готова к сдаче: " + "; ".join(ошибки)
print("Самопроверка пройдена: работа готова к сдаче.")